# Chapitre 3 — ResNet-18 : densité mammaire

**🎯 Objectif :** entraîner un réseau **connu** (ResNet-18 pré-entraîné ImageNet) en transfer learning pour classer la **densité mammaire** (BI-RADS A/B/C/D), avec la recette anti-collapse (LR 1e-5, sampler équilibré, split par patient).
**⏱ Durée :** définition du modèle instantanée ; entraînement de démo ~quelques min sur GPU **si** les données sont présentes (sinon les cellules d'entraînement s'auto-désactivent).

Premier vrai entraînement, avec un réseau **connu** : ResNet-18 pré-entraîné
ImageNet, adapté en **classifieur multiclasse** de la **densité mammaire** (BI-RADS
A / B / C / D). C'est le réflexe de base avant des architectures spécialisées
comme GMIC (ch4).

Ce notebook **entraîne sur les données RSNA** (téléchargées au ch1 dans `data/in/`,
prétraitées au ch2.5 dans `data/work/`). Si elles sont absentes, les cellules
d'entraînement s'auto-désactivent proprement — la définition du modèle, elle, tourne
toujours.

In [ ]:
import os, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
import cv2
from course_utils import flowchart, data_in, data_work

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Localiser train.csv + un dossier d'images : entrées brutes dans data/in/,
# prétraitements (ch2.5) dans data/work/.
TRAIN_CSV = next((p for p in [data_in('rsna', 'train.csv'),
                              data_in('train.csv')] if os.path.isfile(p)), None)
IMG_DIR = next((d for d in [data_work('preprocess_image', 'cropped_images'),
                            data_in('rsna', 'train_images_png')]
                if os.path.isdir(d)), None)
DATA_OK = bool(TRAIN_CSV and IMG_DIR)
print('Device   :', DEVICE)
print('train.csv:', TRAIN_CSV)
print('images   :', IMG_DIR)
print('DATA_OK  :', DATA_OK, '' if DATA_OK else '-> entraînement désactivé (placez les données dans data/in/ ou data/work/)')

flowchart([
    'ResNet-18 pre-entraine ImageNet',
    'conv1 adapte (1 canal) + fc -> 4 classes (densite A/B/C/D)',
    'Dataset : PNG + label densite (train.csv)',
    'Sampler equilibre + CrossEntropyLoss + LR 1e-5',
    'Boucle : forward -> loss -> backward -> step',
], title='Ch3 — ResNet-18 densite')

## Le modèle

Une mammographie est en **niveaux de gris** (1 canal), or ResNet-18 attend 3 canaux
RGB. On remplace la première convolution par une version 1-canal, et la dernière
couche `fc` par une sortie à **4 classes**. On garde les poids ImageNet ailleurs
(transfer learning).

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def build_model(n_classes=4):
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    # conv1 : 3 canaux -> 1 canal (moyenne des poids RGB pré-entraînés)
    w = m.conv1.weight.data.mean(dim=1, keepdim=True)
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.conv1.weight.data = w
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

model = build_model().to(DEVICE)
print('ResNet-18 (1 canal, 4 classes) :', f'{sum(p.numel() for p in model.parameters())/1e6:.1f} M params')
# Test forward sur une image factice
with torch.no_grad():
    out = model(torch.randn(2, 1, 512, 512, device=DEVICE))
print('sortie logits :', tuple(out.shape), '(batch, 4 classes)')

## Le dataset (densité)

On lit `train.csv`, on mappe la densité `A/B/C/D → 0/1/2/3`, et on charge le PNG
correspondant (redimensionné 512×512, normalisé). On ne garde que les lignes dont
l'image existe et la densité est renseignée.

In [ ]:
from torch.utils.data import Dataset

DENS = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

def img_path(pid, iid):
    for cand in [os.path.join(IMG_DIR, str(pid), f'{iid}.png'),
                 os.path.join(IMG_DIR, f'{pid}_{iid}.png')]:
        if os.path.isfile(cand):
            return cand
    return None

class DensityDataset(Dataset):
    def __init__(self, rows, size=512):
        self.rows, self.size = rows, size
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        path, label = self.rows[i][0], self.rows[i][1]   # rows = (path, label, patient_id)
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32)
        img = cv2.resize(img, (self.size, self.size))
        img = (img - img.mean()) / max(img.std(), 1e-5)        # z-score
        return torch.tensor(img[None], dtype=torch.float32), label

samples, labels = [], []
if DATA_OK:
    dfc = pd.read_csv(TRAIN_CSV)
    dfc = dfc[dfc['density'].isin(DENS)]
    for pid, iid, dens in dfc[['patient_id', 'image_id', 'density']].itertuples(index=False):
        p = img_path(pid, iid)
        if p:
            samples.append((p, DENS[dens], pid)); labels.append(DENS[dens])   # on garde patient_id
        if len(samples) >= 4000:    # sous-ensemble pour une démo rapide
            break
    print('échantillons :', len(samples), '| répartition classes :', np.bincount(labels, minlength=4).tolist())
else:
    print('Données absentes -> dataset vide (cellule d entraînement sautée).')

## La recette anti-collapse

Leçons du projet (à ne pas refaire) :

- **LR 1e-5** en fine-tuning (un LR trop haut → *collapse* : toutes les prédictions
  deviennent constantes) ;
- **`WeightedRandomSampler`** pour équilibrer des classes inégales — et dans ce cas
  **`CrossEntropyLoss` simple** (ne PAS cumuler sampler + loss pondérée) ;
- **split au niveau patient** (jamais image) : deux vues d'un même patient
  réparties entre train et test gonflent le score (fuite de données) ;
- pas de `channels_last` avec une entrée **1 canal** (erreur CUDA).

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

if DATA_OK and samples:
    # Split AU NIVEAU PATIENT (pas image) : toutes les vues d'un même patient_id
    # tombent du même côté -> évite la fuite de données (data leakage).
    rng = np.random.default_rng(42)
    pids = np.array(sorted({r[2] for r in samples})); rng.shuffle(pids)
    cut = int(0.8 * len(pids)); train_pids = set(pids[:cut].tolist())
    tr_rows = [r for r in samples if r[2] in train_pids]
    te_rows = [r for r in samples if r[2] not in train_pids]
    tr_labels = np.array([r[1] for r in tr_rows])
    assert train_pids.isdisjoint({r[2] for r in te_rows}), 'fuite patient !'

    # sampler équilibré : poids inversement proportionnel à la fréquence de classe
    freq = np.bincount(tr_labels, minlength=4)
    w = (1.0 / np.maximum(freq, 1))[tr_labels]
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), replacement=True)

    tr = DataLoader(DensityDataset(tr_rows), batch_size=16, sampler=sampler, num_workers=2)
    te = DataLoader(DensityDataset(te_rows), batch_size=16, shuffle=False, num_workers=2)
    opt = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(2):                      # démo courte ; monter pour un vrai run
        model.train()
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in te:
                pred = model(x.to(DEVICE)).argmax(1).cpu()
                correct += (pred == y).sum().item(); total += len(y)
        print(f'epoch {epoch}  loss={loss.item():.3f}  test_acc={correct/total:.3f}')
else:
    print('Entraînement sauté (DATA_OK =', DATA_OK, ').')

## Sur machine puissante : qui décode, qui augmente ?

Un pas d'entraînement mêle du travail **CPU** (lire et **décoder** les images, les augmenter) et du travail **GPU** (forward/backward). Le `DataLoader` prépare les batches avec des *workers* CPU pendant que le GPU calcule. Mais si le GPU est rapide et les images grosses, les workers CPU **n'arrivent pas à suivre** : le GPU attend les données et reste sous-utilisé — le pipeline est *CPU/IO-bound*.

Astuce : laisser les workers CPU ne faire **que le décodage** (incompressible) et déplacer l'**augmentation sur le GPU**, appliquée au **batch entier**. Les transformations de [kornia](https://kornia.readthedocs.io) sont de simples `nn.Module` : on les met sur le GPU comme un modèle. Le GPU étant massivement parallèle, augmenter un batch y est quasi gratuit, et le CPU est déchargé. Sur le fine-tuning GMIC de ce projet, ça a fait passer l'utilisation GPU de **49 % à 91 %** (voir chapitre 5).

Mais l'augmentation seule peut être trompeuse : ce qui compte, c'est le **débit du step complet**. On mesure donc la **pipeline entière** (décodage des **vraies images** → augmentation → forward/backward ResNet-18), en comparant **CPU-aug** (augmentation dans les workers) vs **GPU-aug** (workers décodent seulement, kornia augmente le batch sur GPU), **balayé sur plusieurs tailles de batch**.

> ⚠️ Le résultat **dépend de ta machine** (GPU, CPU, résolution, batch) : le notebook n'affirme rien, il **mesure** et conclut d'après les chiffres. Sur un petit GPU ou un petit batch, le CPU peut très bien gagner.

In [ ]:
# === Benchmark PIPELINE COMPLÈTE (ResNet-18 densité) : décodage réel -> augmentation -> forward/backward ===
# Par taille de batch, on compare l'augmentation CPU (dans les workers) vs GPU (kornia sur le batch).
# NON-déterministe : on rapporte les ms/step et img/s MESURÉS — le verdict dépend de ta machine.
import time, glob, os
import numpy as np, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import kornia.augmentation as K
from course_utils import data_work, data_in

# ---- Paramètres balayables ----
BATCH_SIZES = [2, 4, 8, 16]   # <- balaye ici
IMG_SIZE    = 512             # résolution (512 raisonnable pour 4 Go ; tente 1024)
NUM_WORKERS = 4
N_WARMUP, N_STEPS = 3, 15

# ---- Source = tes images réelles : PNG prétraités (ch2.5), sinon DICOM (ch1) ----
def _find_images():
    for d in [data_work('preprocess_sample', 'cropped_images'),
              data_work('preprocess_image', 'cropped_images')]:
        p = sorted(glob.glob(os.path.join(d, '**', '*.png'), recursive=True))
        if p:
            return p, 'png'
    p = sorted(glob.glob(os.path.join(data_in('rsna_sample'), '**', '*.dcm'), recursive=True))
    return (p, 'dcm') if p else ([], None)

_paths, _kind = _find_images()
assert _paths, "Aucune image — exécute le ch1 (download) et idéalement le ch2.5 (preprocessing)."
print(f"{len(_paths)} images réelles ({_kind}) | device {DEVICE} | img {IMG_SIZE}² | workers {NUM_WORKERS}")

def _load(path):
    if _kind == 'dcm':
        import pydicom
        ds = pydicom.dcmread(path); a = ds.pixel_array.astype(np.float32)
        if ds.PhotometricInterpretation == 'MONOCHROME1':
            a = a.max() - a
    else:
        a = cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32)
    a = cv2.resize(a, (IMG_SIZE, IMG_SIZE))
    return (a - a.mean()) / max(a.std(), 1e-5)

class BatchAugment(nn.Module):                       # même esprit que GPUAugment (ch5)
    def __init__(self):
        super().__init__()
        self.affine = K.RandomAffine(degrees=15.0, translate=(0.05, 0.05), scale=(0.9, 1.1), p=1.0)
        self.gamma = K.RandomGamma(gamma=(0.8, 1.25), p=0.5)
    def forward(self, x):
        return self.gamma(self.affine(x))

class BenchDataset(Dataset):
    """Décode une vraie image ; si cpu_aug, augmente DANS le worker (CPU)."""
    def __init__(self, n, cpu_aug):
        self.n = n
        self.aug = BatchAugment() if cpu_aug else None
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        x = torch.from_numpy(_load(_paths[i % len(_paths)]))[None]   # (1, H, W)
        if self.aug is not None:
            x = self.aug(x[None])[0]                                 # augmentation CPU (worker)
        return x, i % 4                                              # label bidon (on mesure le temps)

model = build_model().to(DEVICE).train()             # ResNet-18 (1 canal, 4 classes), réutilisé
lossf = nn.CrossEntropyLoss()

def bench(batch_size, mode):
    """mode='cpu' (augmentation dans les workers) ou 'gpu' (kornia sur le batch GPU)."""
    dl = DataLoader(BenchDataset(batch_size * (N_WARMUP + N_STEPS), cpu_aug=(mode == 'cpu')),
                    batch_size=batch_size, num_workers=NUM_WORKERS, shuffle=True, pin_memory=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    gpu_aug = BatchAugment().to(DEVICE) if mode == 'gpu' else None
    t0 = None
    try:
        for step, (x, y) in enumerate(dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            if gpu_aug is not None:
                x = gpu_aug(x)
            opt.zero_grad(); lossf(model(x), y).backward(); opt.step()
            if step == N_WARMUP - 1:                 # fin du warmup -> on démarre le chrono
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        dt = (time.time() - t0) / N_STEPS
        return dt * 1000, batch_size / dt            # ms/step, img/s
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache(); return None, None
        raise

def _f(ms, ips):
    return f"{ms:8.0f} {ips:7.1f}" if ms else f"{'OOM / —':>16}"

print(f"\n{'batch':>5} | {'CPU-aug  ms/step img/s':>22} | {'GPU-aug  ms/step img/s':>22} | verdict")
print('-' * 78)
for bs in BATCH_SIZES:
    c_ms, c_ips = bench(bs, 'cpu')
    g_ms, g_ips = bench(bs, 'gpu') if DEVICE.type == 'cuda' else (None, None)
    if c_ms and g_ms:
        verdict = f"GPU-aug {c_ms / g_ms:.2f}×" if g_ms < c_ms else f"CPU-aug {g_ms / c_ms:.2f}×"
    else:
        verdict = "(pas de GPU)" if DEVICE.type != 'cuda' else "OOM"
    print(f"{bs:5d} | {_f(c_ms, c_ips)} | {_f(g_ms, g_ips)} | {verdict}")

print("\nOn mesure le STEP complet (décodage + augmentation + forward/backward).")
print("Le gain GPU-aug n'apparaît que si l'augmentation CPU sature les workers ; sinon le CPU peut gagner.")

## Récap

- ResNet-18 ImageNet → adapté en 1 canal / 4 classes : transfer learning standard.
- Recette anti-collapse : **LR 1e-5**, sampler équilibré **ou** loss pondérée (pas
  les deux), CrossEntropy.
- Un ResNet image-level **plafonne** sur des tâches fines (petites lésions) car le
  pooling global dilue les détails → d'où GMIC, qui travaille en pleine résolution.

Chapitre suivant → `04_gmic_architecture.ipynb`.

## 🧪 Smoke test

Vérifie que le code clé de ce chapitre tourne **sans télécharger les données ni lancer le preprocessing complet** (entrées synthétiques).

In [ ]:
# 🧪 Smoke test — valide modèle, dataset et split-par-patient SANS données réelles.
import os, tempfile, numpy as np, torch, cv2

# 1. build_model : forward + 4 logits en sortie
_m = build_model(n_classes=4).eval()
with torch.no_grad():
    _out = _m(torch.randn(2, 1, 256, 256, device=next(_m.parameters()).device))
assert _out.shape == (2, 4), _out.shape

# 2. DensityDataset sur un faux PNG (aucun téléchargement / preprocessing)
with tempfile.TemporaryDirectory() as d:
    _p = os.path.join(d, 'fake.png')
    cv2.imwrite(_p, (np.random.rand(64, 80) * 255).astype('uint8'))
    _rows = [(_p, 2, 'patient_X')]            # (path, label, patient_id)
    _img, _lbl = DensityDataset(_rows, size=128)[0]
    assert tuple(_img.shape) == (1, 128, 128) and _lbl == 2, (_img.shape, _lbl)

# 3. Split AU NIVEAU PATIENT : aucune fuite train/test (3 vues par patient)
_samples = [(f'/x/{i}.png', i % 4, f'pat{i // 3}') for i in range(60)]
_rng = np.random.default_rng(0)
_pids = np.array(sorted({r[2] for r in _samples})); _rng.shuffle(_pids)
_cut = int(0.8 * len(_pids)); _train = set(_pids[:_cut].tolist())
_tr = {r[2] for r in _samples if r[2] in _train}
_te = {r[2] for r in _samples if r[2] not in _train}
assert _tr.isdisjoint(_te), 'fuite patient détectée !'
print('✅ build_model, DensityDataset et split-par-patient (sans fuite) OK, sans données.')